# ESG Climate Commitment RAG (Colab-Friendly Teaching Notebook)

In this notebook, We learn how RAG works through a practical climate-commitment assessment workflow.

We build a **simple RAG pipeline** to evaluate climate commitments in ESG / sustainability reports for:
- Amazon
- Google
- Apple
- Meta

## Colab Quick Start
1. We open this notebook in Google Colab.
2. We run the Colab setup cells first.
3. We use Colab's built-in AI model (`google.colab.ai`) for generation, so We do not need an external API key.
4. We run one company first, then the four-company batch.

## Teaching goal
We use this flow:
1. Parse report text
2. Split into chunks
3. Store chunks in a vector database
4. Retrieve relevant chunks for assessment questions
5. Ask Colab's built-in LLM to perform TCFD-aligned climate commitment scoring


## Practical Context: Green Finance / ESG Review

When exact emissions values are hard to extract from complex tables,
We can still evaluate disclosure quality and commitment credibility.

In this notebook, We focus on:
- Governance quality
- Strategy and resilience quality
- Risk-management integration
- Metrics and targets quality
- Transition plan credibility


    ## Dependencies (Colab-Friendly Setup)

    For Google Colab, We run the setup cells below instead of manually installing everything.

    Important:
    - We use Colab's built-in AI model (`google.colab.ai`) for text generation (no external API key).
    - We install local packages for parsing PDFs, building embeddings, and running ChromaDB retrieval.
    - We usually keep Colab's preinstalled `torch` unchanged.
    - This notebook is tested locally with Python `3.12.10`, and this Colab version is designed to run in Colab's managed Python environment.
    


    ## Colab Runtime Setup (Colab AI + Packages)

    We run this section first in Colab.

    - The LLM call uses Colab's built-in AI service (`google.colab.ai`), so We do not configure an API key.
    - We keep Colab's preinstalled `torch`.
    - We install the remaining packages (`chromadb`, `sentence-transformers`, etc.) for the local RAG pipeline.
    - GPU is optional here. Colab AI generation does not depend on our local GPU, but GPU can still help embeddings run faster.
    


In [ ]:
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
RUN_COLAB_INSTALL = True

print("IN_COLAB =", IN_COLAB)
print("Python version =", sys.version.split()[0])

if IN_COLAB and RUN_COLAB_INSTALL:
    packages = [
        "pypdf==6.7.3",
        "pandas==3.0.1",
        "chromadb==1.5.1",
        "sentence-transformers==5.2.3",
        "python-dotenv==1.2.1",
        "requests==2.32.5",
    ]
    print("Installing Colab packages (keeping preinstalled torch)...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
    print("Colab package installation complete.")
else:
    print("Skipping Colab package installation.")

if IN_COLAB:
    print("\nColab AI note: This notebook uses google.colab.ai (no external API key).")



In [ ]:
COLAB_AI_AVAILABLE = False
COLAB_AI_IMPORT_ERROR = None

try:
    import torch
    print("torch version =", torch.__version__)
    print("CUDA available =", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU =", torch.cuda.get_device_name(0))
    else:
        print("GPU is optional for this notebook. Colab AI generation still works without a local GPU.")
except Exception as e:
    print("Torch check skipped:", e)

if IN_COLAB:
    try:
        from google.colab import ai as _colab_ai_probe  # noqa: F401
        COLAB_AI_AVAILABLE = True
        print("google.colab.ai is available in this runtime.")
    except Exception as e:
        COLAB_AI_IMPORT_ERROR = str(e)
        print("google.colab.ai is not available:", e)
        print("Open this notebook in Google Colab and use a standard Colab runtime.")
else:
    print("Not running in Colab. google.colab.ai is not available in local notebooks.")



## Data Folder (Expected)

```text
RAG/
  simple_rag_esg_carbon_teaching_colab.ipynb
  data/
    esg_reports/
      amazon_2024_sustainability_report.pdf
      apple_2025_environmental_progress_report.pdf
      google_2025_environmental_report.pdf
      meta_2025_sustainability_report.pdf
```

Note:
- The parser in this notebook supports both `.pdf` and `.md/.txt`.


## Optional: Persist Files in Google Drive (Colab)

Colab local storage (`/content`) is temporary.
If we want to keep downloaded reports, vector DB files, and outputs after the session ends, we can mount Google Drive and set `PROJECT_ROOT`.


In [ ]:
USE_GOOGLE_DRIVE = False
GOOGLE_DRIVE_PROJECT_DIR = "/content/drive/MyDrive/ESG_RAG_colab"
import sys

if "google.colab" in sys.modules and USE_GOOGLE_DRIVE:
    from google.colab import drive
    import os
    drive.mount("/content/drive")
    os.makedirs(GOOGLE_DRIVE_PROJECT_DIR, exist_ok=True)
    os.environ["PROJECT_ROOT"] = GOOGLE_DRIVE_PROJECT_DIR
    print("PROJECT_ROOT set to:", os.environ["PROJECT_ROOT"])
else:
    print("Using current working directory. PROJECT_ROOT is not overridden.")


In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import sys
import re
import json
import time
from datetime import datetime
from typing import Any, Dict, List

import pandas as pd
import requests
from pypdf import PdfReader
from dotenv import load_dotenv

import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

load_dotenv()

IN_COLAB = "google.colab" in sys.modules
PROJECT_ROOT = Path(os.getenv("PROJECT_ROOT", ".")).resolve()
DATA_DIR = PROJECT_ROOT / "data" / "esg_reports"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
VECTOR_DB_DIR = PROJECT_ROOT / "vector_db" / "chroma_esg"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
VECTOR_DB_DIR.mkdir(exist_ok=True, parents=True)

print("IN_COLAB:", IN_COLAB)
print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR.resolve())
print("Output dir:", OUTPUT_DIR.resolve())
print("Vector DB dir:", VECTOR_DB_DIR.resolve())



## Optional Download Helper (for reproducibility)

We use these URLs in this run.
In Colab, we keep the remote download option and default to downloading missing files automatically.
We can still set `DOWNLOAD_MISSING = False` if the files are already available.

Important:
- We use official report links for the four target companies in this notebook run.


In [ ]:
DOWNLOAD_MISSING = bool(globals().get("IN_COLAB", False))
# Colab default: download missing reports automatically.
# Local default: keep False unless we want to download.
# We can override manually, for example:
# DOWNLOAD_MISSING = False

SOURCE_URLS = {
    "amazon_2024_sustainability_report.pdf": "https://sustainability.aboutamazon.com/2024-amazon-sustainability-report.pdf",
    "apple_2025_environmental_progress_report.pdf": "https://www.apple.com/environment/pdf/Apple_Environmental_Progress_Report_2025.pdf",
    "google_2025_environmental_report.pdf": "https://www.gstatic.com/gumdrop/sustainability/google-2025-environmental-report.pdf",
    "meta_2025_sustainability_report.pdf": "https://sustainability.atmeta.com/wp-content/uploads/2025/08/Meta_2025-Sustainability-Report_.pdf",
}

if DOWNLOAD_MISSING:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    headers = {"User-Agent": "Mozilla/5.0"}
    for name, url in SOURCE_URLS.items():
        path = DATA_DIR / name
        if path.exists() and path.stat().st_size > 100_000:
            print("SKIP", name)
            continue
        print("Downloading", name)
        r = requests.get(url, headers=headers, timeout=300)
        r.raise_for_status()
        if name.endswith(".pdf"):
            path.write_bytes(r.content)
        else:
            path.write_text(r.text, encoding="utf-8")
        print("Saved", name, path.stat().st_size, "bytes")
else:
    print("DOWNLOAD_MISSING = False (using local files if present)")

with open(OUTPUT_DIR / "download_sources.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "generated_at": datetime.utcnow().isoformat() + "Z",
            "source_urls": SOURCE_URLS,
        },
        f,
        indent=2,
        ensure_ascii=False,
    )
print("Saved source URL record to", OUTPUT_DIR / "download_sources.json")


## Step 1. Load Documents (PDF + Text Mirror Support)

We keep parsing lightweight:
- `pypdf` for normal PDFs
- direct text loading for `.md/.txt`

For text mirror files, we remove the `r.jina.ai` header and keep the report content section.


In [ ]:
def extract_text_from_pdf(pdf_path: Path) -> str:
    reader = PdfReader(str(pdf_path))
    parts = []
    for page in reader.pages:
        parts.append(page.extract_text() or "")
    return "\n".join(parts)


def extract_text_from_text_file(path: Path) -> str:
    text = path.read_text(encoding="utf-8", errors="ignore")
    # r.jina.ai PDF mirror format includes a header and then "Markdown Content:"
    marker = "Markdown Content:"
    if marker in text:
        text = text.split(marker, 1)[1].strip()
    return text


def infer_company_label(file_name: str) -> str:
    lower = file_name.lower()
    if lower.startswith("amazon_"):
        return "Amazon"
    if lower.startswith("google_"):
        return "Google"
    if lower.startswith("apple_"):
        return "Apple"
    if lower.startswith("meta_"):
        return "Meta"
    return Path(file_name).stem


def load_documents(data_dir: Path) -> List[Dict[str, Any]]:
    docs: List[Dict[str, Any]] = []
    for path in sorted(data_dir.iterdir()):
        if path.name.lower().startswith("tesla_"):
            continue
        if path.suffix.lower() not in {".pdf", ".md", ".txt"}:
            continue
        try:
            if path.suffix.lower() == ".pdf":
                text = extract_text_from_pdf(path)
                source_type = "pdf"
            else:
                text = extract_text_from_text_file(path)
                source_type = "text_mirror"
        except Exception as e:
            print(f"SKIP (parse error): {path.name} -> {e}")
            continue

        docs.append(
            {
                "doc_id": path.stem,
                "company": infer_company_label(path.name),
                "file_name": path.name,
                "source_type": source_type,
                "text": text,
                "char_count": len(text),
            }
        )
    return docs


documents = load_documents(DATA_DIR) if DATA_DIR.exists() else []
print(f"Loaded {len(documents)} document(s).")
for d in documents:
    print(f"- {d['company']}: {d['file_name']} ({d['source_type']}) | {d['char_count']:,} chars")


## Step 2. Chunk the Documents

We split report text into chunks before storing them in the vector database.

For faster embedding in Colab, We use a speed-focused chunk setup:
- larger chunks (fewer chunks total)
- smaller overlap
- same simple paragraph-aware chunker


In [ ]:
def normalize_text(text: str) -> str:
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


# Speed-focused chunking defaults for Colab:
# - larger chunks -> fewer embeddings to compute
# - smaller overlap -> less duplicated text across chunks
CHUNK_MAX_CHARS = 1400
CHUNK_OVERLAP_CHARS = 60


def chunk_text(
    text: str,
    max_chars: int = CHUNK_MAX_CHARS,
    overlap_chars: int = CHUNK_OVERLAP_CHARS,
) -> List[str]:
    text = normalize_text(text)
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks: List[str] = []
    current = ""

    for para in paragraphs:
        candidate = f"{current}\n\n{para}".strip() if current else para
        if len(candidate) <= max_chars:
            current = candidate
            continue

        if current:
            chunks.append(current)
            current = ""

        if len(para) <= max_chars:
            current = para
        else:
            start = 0
            while start < len(para):
                end = start + max_chars
                piece = para[start:end]
                chunks.append(piece)
                if end >= len(para):
                    break
                start = max(0, end - overlap_chars)

    if current:
        chunks.append(current)

    # lightweight overlap between adjacent chunks
    if overlap_chars > 0 and len(chunks) > 1:
        out = [chunks[0]]
        for i in range(1, len(chunks)):
            prefix = chunks[i - 1][-overlap_chars:]
            out.append((prefix + "\n" + chunks[i]).strip())
        chunks = out

    return chunks


def build_chunk_records(
    docs: List[Dict[str, Any]],
    max_chars: int = CHUNK_MAX_CHARS,
    overlap_chars: int = CHUNK_OVERLAP_CHARS,
) -> List[Dict[str, Any]]:
    records: List[Dict[str, Any]] = []
    for doc in docs:
        chunks = chunk_text(doc["text"], max_chars=max_chars, overlap_chars=overlap_chars)
        for i, chunk in enumerate(chunks):
            records.append(
                {
                    "id": f"{doc['doc_id']}::chunk::{i}",
                    "doc_id": doc["doc_id"],
                    "company": doc["company"],
                    "file_name": doc["file_name"],
                    "source_type": doc["source_type"],
                    "chunk_id": i,
                    "text": chunk,
                }
            )
    return records


chunk_records = build_chunk_records(
    documents,
    max_chars=CHUNK_MAX_CHARS,
    overlap_chars=CHUNK_OVERLAP_CHARS,
)
print("Chunk config:", {"max_chars": CHUNK_MAX_CHARS, "overlap_chars": CHUNK_OVERLAP_CHARS})
print("Total chunks:", len(chunk_records))
if chunk_records:
    print("Example chunk:", chunk_records[0]["id"])
    print(chunk_records[0]["text"][:600])


## Step 3. Build a Vector Database (ChromaDB)

We store chunks in a vector database using:
- `ChromaDB` (local persistent vector DB)
- a smaller embedding model for speed (`paraphrase-MiniLM-L3-v2`)

Speed changes in this version of the Colab notebook:
- default `REBUILD_VECTOR_DB = False` (reuse existing index)
- try GPU for embedding in Colab when CUDA is available
- skip re-indexing when the collection already exists and has data


In [ ]:
EMBED_MODEL_NAME = "sentence-transformers/paraphrase-MiniLM-L3-v2"
COLLECTION_NAME = "esg_climate_commitment_chunks_fast_l3_colab_v1"
REBUILD_VECTOR_DB = False  # build once, then reuse unless We intentionally rebuild


def detect_embedding_device() -> str:
    try:
        import torch
        if IN_COLAB and torch.cuda.is_available():
            return "cuda"
    except Exception:
        pass
    return "cpu"


EMBEDDING_DEVICE = detect_embedding_device()
print("Embedding model:", EMBED_MODEL_NAME)
print("Embedding device:", EMBEDDING_DEVICE)
print("Collection name:", COLLECTION_NAME)
print("REBUILD_VECTOR_DB:", REBUILD_VECTOR_DB)


def make_embedding_function(model_name: str, device: str):
    # Chroma's SentenceTransformerEmbeddingFunction supports `device` in recent versions.
    # We keep a fallback for compatibility with older versions.
    try:
        print(f"Creating embedding function on device={device} ...")
        return SentenceTransformerEmbeddingFunction(model_name=model_name, device=device)
    except TypeError:
        print("This Chroma version does not accept device=... in SentenceTransformerEmbeddingFunction.")
        print("Falling back to default constructor (embedding may run on CPU).")
        return SentenceTransformerEmbeddingFunction(model_name=model_name)


embedding_fn = make_embedding_function(EMBED_MODEL_NAME, EMBEDDING_DEVICE)
chroma_client = chromadb.PersistentClient(path=str(VECTOR_DB_DIR))

if REBUILD_VECTOR_DB:
    try:
        chroma_client.delete_collection(COLLECTION_NAME)
        print("Deleted existing collection:", COLLECTION_NAME)
    except Exception:
        pass

collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"},
)


def index_chunks_to_chroma(collection, chunk_records: List[Dict[str, Any]], batch_size: int = 64) -> None:
    if not chunk_records:
        print("No chunks to index.")
        return

    for start in range(0, len(chunk_records), batch_size):
        batch = chunk_records[start : start + batch_size]
        collection.add(
            ids=[r["id"] for r in batch],
            documents=[r["text"] for r in batch],
            metadatas=[
                {
                    "doc_id": r["doc_id"],
                    "company": r["company"],
                    "file_name": r["file_name"],
                    "source_type": r["source_type"],
                    "chunk_id": int(r["chunk_id"]),
                }
                for r in batch
            ],
        )
    print("Indexed", len(chunk_records), "chunks into ChromaDB.")


collection_count_before = collection.count()
print("Collection count before indexing:", collection_count_before)

if REBUILD_VECTOR_DB or collection_count_before == 0:
    index_chunks_to_chroma(collection, chunk_records)
else:
    print("Reusing existing collection. Skipping indexing because REBUILD_VECTOR_DB = False.")
    print("If We change chunking or embedding settings, We should use a new COLLECTION_NAME or set REBUILD_VECTOR_DB = True.")

print("Collection count:", collection.count())


## Step 4. Retrieval from the Vector DB (Hybrid + Table/Section-Aware Rerank)

We use a hybrid retrieval strategy:
- semantic retrieval query (indicator meaning)
- section/table-oriented retrieval query (governance/risk/targets/progress language)
- rerank by disclosure signals and numeric/table clues

This improves recall for commitment evidence spread across narrative sections and data tables.


In [ ]:
NUMBER_TOKEN_PATTERN = r"\b\d{1,3}(?:,\d{3})*(?:\.\d+)?\b|\b\d+\.\d+\b"


def retrieve_chunks(query: str, top_k: int = 4, filter_doc_id: str | None = None) -> List[Dict[str, Any]]:
    kwargs: Dict[str, Any] = {
        "query_texts": [query],
        "n_results": top_k,
        "include": ["documents", "metadatas", "distances"],
    }
    if filter_doc_id is not None:
        kwargs["where"] = {"doc_id": filter_doc_id}

    res = collection.query(**kwargs)

    docs = res.get("documents", [[]])[0]
    metas = res.get("metadatas", [[]])[0]
    distances = res.get("distances", [[]])[0]

    out = []
    for doc_text, meta, distance in zip(docs, metas, distances):
        item = dict(meta)
        item["text"] = doc_text
        item["distance"] = float(distance)
        # cosine distance -> rough similarity (teaching convenience)
        item["similarity"] = max(0.0, 1.0 - float(distance))
        out.append(item)
    return out


def retrieve_chunks_multiquery(
    query_texts: List[str],
    top_k_per_query: int = 8,
    filter_doc_id: str | None = None,
) -> List[Dict[str, Any]]:
    merged: Dict[str, Dict[str, Any]] = {}
    for q in query_texts:
        if not q.strip():
            continue
        hits = retrieve_chunks(q, top_k=top_k_per_query, filter_doc_id=filter_doc_id)
        for h in hits:
            key = f"{h['file_name']}::{h['chunk_id']}"
            prev = merged.get(key)
            item = dict(h)
            if prev is None:
                item["matched_queries"] = [q[:180]]
                merged[key] = item
            else:
                prev["similarity"] = max(prev["similarity"], item["similarity"])
                prev["distance"] = min(prev["distance"], item["distance"])
                prev.setdefault("matched_queries", [])
                if q[:180] not in prev["matched_queries"]:
                    prev["matched_queries"].append(q[:180])

    out = list(merged.values())
    out.sort(key=lambda x: x["similarity"], reverse=True)
    return out


def compute_table_signal_score(text: str) -> float:
    lower = text.lower()
    score = 0.0
    if re.search(r"\bscope\s*[123]\b", lower):
        score += 1.0
    if re.search(r"co2e|tco2|mtco2|metric tons?|mmt", lower):
        score += 1.0
    if re.search(r"data table|emissions category|gross emissions|totals|fiscal year|yoy", lower):
        score += 1.0
    if "2024" in lower and ("2023" in lower or "2022" in lower):
        score += 0.8
    number_count = len(re.findall(NUMBER_TOKEN_PATTERN, text))
    if number_count >= 6:
        score += 1.0
    elif number_count >= 3:
        score += 0.5
    return score


def looks_table_like(text: str) -> bool:
    return compute_table_signal_score(text) >= 2.0


example_query = "climate governance strategy risk management targets progress transition plan resilience"
if documents:
    example_doc_id = documents[0]["doc_id"]
    hits = retrieve_chunks_multiquery(
        query_texts=[
            example_query,
            "governance committee risk management scenario analysis climate targets progress milestones",
        ],
        top_k_per_query=5,
        filter_doc_id=example_doc_id,
    )
    print("Retrieved chunks for:", example_doc_id)
    for i, h in enumerate(hits[:5], 1):
        print(
            f"\n[{i}] {h['file_name']} | chunk={h['chunk_id']} | "
            f"similarity={h['similarity']:.4f} | table_signal={compute_table_signal_score(h['text']):.2f}"
        )
        print(h["text"][:500])
else:
    print("No documents found.")


    ## Step 5. Use Colab Built-In AI Model (No API Key)

    We keep retrieval fixed and call Colab's built-in AI model for generation.

    Core interface (from the official Colab AI notebook):
    - `from google.colab import ai`
    - `ai.list_models()`
    - `ai.generate_text(prompt, model_name=...)`

    This keeps the teaching flow focused on RAG, not on API account setup.
    


In [ ]:
_COLAB_AI_CACHE: Dict[str, Any] = {}


def get_colab_ai_module():
    if "module" in _COLAB_AI_CACHE:
        return _COLAB_AI_CACHE["module"]

    if "google.colab" not in sys.modules:
        raise RuntimeError(
            "This notebook path uses google.colab.ai. "
            "Open the notebook in Google Colab to run the LLM step."
        )

    from google.colab import ai as colab_ai

    _COLAB_AI_CACHE["module"] = colab_ai
    return colab_ai


def _response_to_text(resp: Any) -> str:
    if isinstance(resp, str):
        return resp.strip()
    if hasattr(resp, "text") and isinstance(resp.text, str):
        return resp.text.strip()
    if isinstance(resp, dict):
        for key in ["text", "output_text", "response", "content"]:
            val = resp.get(key)
            if isinstance(val, str):
                return val.strip()
    return str(resp).strip()


def list_colab_ai_models(limit: int = 20):
    colab_ai = get_colab_ai_module()
    models = colab_ai.list_models()
    print("Available models (showing up to", limit, "):")
    try:
        for i, m in enumerate(models[:limit], 1):
            if isinstance(m, dict):
                name = m.get("name") or m.get("model_name") or str(m)
            else:
                name = getattr(m, "name", str(m))
            print(f"{i:02d}. {name}")
    except Exception:
        print(models)
    return models


def _is_model_unavailable_error(e: Exception) -> bool:
    msg = str(e).lower()
    return ("unavailable" in msg and "model" in msg) or "503" in msg


def _unique_keep_order(items: List[str]) -> List[str]:
    seen = set()
    out = []
    for x in items:
        if not x or x in seen:
            continue
        seen.add(x)
        out.append(x)
    return out


def call_colab_ai_llm(
    system_prompt: str,
    user_prompt: str,
    model_name: str | None = None,
) -> str:
    colab_ai = get_colab_ai_module()
    primary = model_name or os.getenv("COLAB_AI_MODEL_NAME", "google/gemini-2.5-pro")
    fallback_models = list(globals().get("COLAB_AI_FALLBACK_MODELS", []))
    candidate_models = _unique_keep_order([primary, *fallback_models])

    combined_prompt = (
        "System instructions:\n"
        f"{system_prompt}\n\n"
        "User request:\n"
        f"{user_prompt}\n\n"
        "Return JSON only."
    )

    last_error = None
    _COLAB_AI_CACHE["last_model_used"] = None
    _COLAB_AI_CACHE["last_model_attempts"] = []

    for candidate in candidate_models:
        try:
            resp = colab_ai.generate_text(combined_prompt, model_name=candidate)
            _COLAB_AI_CACHE["last_model_used"] = candidate
            _COLAB_AI_CACHE["last_model_attempts"].append({"model": candidate, "status": "ok"})
            if candidate != primary:
                print(f"[Colab AI] Fallback activated: {primary} -> {candidate}")
            return _response_to_text(resp)
        except Exception as e:
            _COLAB_AI_CACHE["last_model_attempts"].append(
                {"model": candidate, "status": "error", "error": str(e)[:300]}
            )
            last_error = e
            if _is_model_unavailable_error(e):
                print(f"[Colab AI] Model unavailable: {candidate}. Trying next fallback...")
                continue
            raise

    if last_error is not None:
        raise last_error
    raise RuntimeError("No Colab AI models were configured for generation.")


def call_llm(system_prompt: str, user_prompt: str, model_name: str | None = None) -> str:
    return call_colab_ai_llm(system_prompt=system_prompt, user_prompt=user_prompt, model_name=model_name)


## Step 6. Climate Commitment Scoring (TCFD + IFRS S2 Aligned)

When exact emissions numbers are difficult to extract, We can still evaluate disclosure quality and commitment credibility.

We score each company on five indicators aligned to TCFD pillars and IFRS S2 climate disclosure expectations:
1. Governance Oversight
2. Strategy & Resilience
3. Risk Management Integration
4. Metrics & Targets Quality
5. Transition Plan Credibility

Scoring structure:
- Indicator score: `0-4`
- Weighted base score: `0-100`
- Quality adjustment: evidence sufficiency + semantic risk checks
- Final score and grade: adjusted `0-100` with `AAA` to `CCC`


### Framework References Used in This Rubric

- TCFD Recommendations (four pillars + 11 recommended disclosures): https://www.fsb-tcfd.org/recommendations/
- TCFD guidance library (strategy, risk, metrics, scenario analysis): https://www.fsb-tcfd.org/publications/
- IFRS S2 Climate-related Disclosures (built on TCFD architecture): https://www.ifrs.org/issued-standards/ifrs-sustainability-standards-navigator/ifrs-s2-climate-related-disclosures/
- IFRS Foundation TCFD materials transition page (monitoring transfer context): https://www.ifrs.org/projects/completed-projects/2024/tcfd-materials/
- GHG Protocol Corporate Standard: https://ghgprotocol.org/corporate-standard
- GHG Protocol Scope 3 Standard: https://ghgprotocol.org/corporate-value-chain-scope-3-standard


In [ ]:
COMMITMENT_SYSTEM_PROMPT = """
You are an ESG disclosure reviewer focused on climate-related reporting quality.
Evaluate disclosures with TCFD structure and IFRS S2 expectations.
Use only retrieved excerpts, score conservatively, and avoid unsupported assumptions.
Do not reward aspirational statements without implementation evidence.
Return one valid JSON object only.
""".strip()

TCFD_DISCLOSURE_QUALITY_PRINCIPLES = [
    "Represent relevant information.",
    "Be specific and complete.",
    "Be clear, balanced, and understandable.",
    "Be consistent over time.",
    "Be comparable among peers within the sector.",
    "Be reliable, verifiable, and objective.",
    "Be provided on a timely basis.",
]

TCFD_COMMITMENT_RUBRIC: Dict[str, Dict[str, Any]] = {
    "governance_oversight": {
        "tcfd_ref": "TCFD Governance (a,b)",
        "ifrs_s2_ref": "IFRS S2 Governance disclosures",
        "weight": 0.15,
        "what_to_check": "Board and management oversight architecture for climate topics, with decision rights and accountability.",
        "required_disclosures": [
            "Board or committee oversight role for climate-related matters",
            "Management role and execution responsibilities",
            "Oversight cadence (for example quarterly/annual reviews)",
            "Accountability linkage such as incentives, KPIs, or escalation",
        ],
        "over_scoring_red_flags": [
            "Only generic tone-at-the-top language",
            "No clear owner or review cadence",
            "No evidence of accountability mechanism",
        ],
        "score_levels": {
            0: "No meaningful governance disclosure.",
            1: "Mentions governance at a high level but no role clarity.",
            2: "Some role disclosure but incomplete oversight cadence/accountability.",
            3: "Clear board and management roles with periodic oversight evidence.",
            4: "Strong governance architecture with role clarity, cadence, and accountability linkage.",
        },
        "retrieval_terms": [
            "board",
            "committee",
            "oversight",
            "management",
            "governance",
            "accountability",
            "executive compensation",
            "climate governance",
        ],
    },
    "strategy_resilience": {
        "tcfd_ref": "TCFD Strategy (a,b,c)",
        "ifrs_s2_ref": "IFRS S2 climate resilience and strategy disclosures",
        "weight": 0.20,
        "what_to_check": "Integration of climate risks/opportunities into strategy and resilience under climate scenarios.",
        "required_disclosures": [
            "Key climate-related risks and opportunities tied to business lines",
            "How strategy responds to transition and/or physical risks",
            "Scenario analysis references (assumptions, pathways, or scenario names)",
            "Resilience implications for business model or capital allocation",
        ],
        "over_scoring_red_flags": [
            "Ambition language without scenario context",
            "No explicit linkage to strategy or business model",
            "No mention of uncertainty, constraints, or trade-offs",
        ],
        "score_levels": {
            0: "No climate strategy or resilience discussion.",
            1: "General strategy language with no decision-useful detail.",
            2: "Partial strategy linkage with limited resilience/scenario detail.",
            3: "Clear strategy linkage with scenario/resilience evidence and implications.",
            4: "Robust resilience narrative with scenario assumptions and strategic responses.",
        },
        "retrieval_terms": [
            "strategy",
            "scenario",
            "resilience",
            "transition",
            "physical risk",
            "opportunity",
            "business model",
            "capital allocation",
        ],
    },
    "risk_management_integration": {
        "tcfd_ref": "TCFD Risk Management (a,b,c)",
        "ifrs_s2_ref": "IFRS S2 risk management process integration",
        "weight": 0.20,
        "what_to_check": "How climate risk identification, assessment, prioritization, and monitoring integrate into enterprise risk management.",
        "required_disclosures": [
            "Process to identify climate-related risks",
            "Assessment/prioritization criteria or materiality logic",
            "Monitoring and control process (thresholds, triggers, reviews)",
            "Integration with enterprise risk management and governance",
        ],
        "over_scoring_red_flags": [
            "Mentions risk register without process detail",
            "No prioritization logic",
            "No evidence of integration with core risk framework",
        ],
        "score_levels": {
            0: "No risk-management process for climate topics.",
            1: "Mentions climate risk but process is vague.",
            2: "Partial process description with weak integration evidence.",
            3: "Defined process and clear integration into broader risk management.",
            4: "Comprehensive process with prioritization logic and governance linkage.",
        },
        "retrieval_terms": [
            "risk management",
            "identify",
            "assess",
            "prioritize",
            "materiality",
            "ERM",
            "controls",
            "risk appetite",
        ],
    },
    "metrics_targets_quality": {
        "tcfd_ref": "TCFD Metrics & Targets (a,b,c)",
        "ifrs_s2_ref": "IFRS S2 metrics and targets (including Scope 1, 2, 3 where relevant)",
        "weight": 0.25,
        "what_to_check": "Quality of emissions metrics and targets: boundary, baseline, methodology, progress, and consistency.",
        "required_disclosures": [
            "At least one quantified climate metric (for example emissions, intensity, energy)",
            "Target boundary and baseline year",
            "Target horizon and interim milestones",
            "Methodological notes (for example GHG Protocol, market/location-based notes)",
            "Progress tracking versus baseline or prior year",
        ],
        "over_scoring_red_flags": [
            "Net-zero claim without baseline, scope boundary, or timeline",
            "Inconsistent numbers across sections without explanation",
            "No methodology or restatement note when numbers change",
        ],
        "score_levels": {
            0: "No useful metrics/targets disclosure.",
            1: "Target claims with minimal quantification and no boundary clarity.",
            2: "Some quantified metrics/targets but major boundary or method gaps.",
            3: "Good metrics package with baseline, timeline, and progress evidence.",
            4: "High-quality metrics package with transparent methodology and consistent progress tracking.",
        },
        "retrieval_terms": [
            "scope 1",
            "scope 2",
            "scope 3",
            "target",
            "baseline",
            "progress",
            "methodology",
            "ghg protocol",
            "market-based",
            "location-based",
            "intensity",
        ],
    },
    "transition_plan_credibility": {
        "tcfd_ref": "TCFD Strategy (c) + Metrics & Targets (c)",
        "ifrs_s2_ref": "IFRS S2 transition planning and execution disclosures",
        "weight": 0.20,
        "what_to_check": "Execution credibility of transition plan through actions, resources, milestones, and delivery evidence.",
        "required_disclosures": [
            "Concrete decarbonization levers (operations, product, value chain)",
            "Milestones or timeline with delivery checkpoints",
            "Resource commitment signals (for example capex, procurement, program scale)",
            "Execution evidence and constraints/risks",
        ],
        "over_scoring_red_flags": [
            "Roadmap language without dated milestones",
            "No resource commitment signal",
            "No evidence of implemented actions",
        ],
        "score_levels": {
            0: "No practical transition plan evidence.",
            1: "Aspirational narrative with little implementation detail.",
            2: "Some actions listed but weak milestones/resources/accountability.",
            3: "Credible plan with concrete actions and progress indicators.",
            4: "Highly credible plan with milestones, resources, delivery evidence, and transparent constraints.",
        },
        "retrieval_terms": [
            "transition plan",
            "roadmap",
            "milestone",
            "investment",
            "capex",
            "implementation",
            "supplier engagement",
            "abatement",
            "decarbonization",
        ],
    },
}

COMMITMENT_OUTPUT_SCHEMA = {
    "company": "string",
    "report_year_focus": "string or null",
    "indicator_scores": {
        "governance_oversight": {
            "score_0_to_4": "number",
            "rating": "insufficient|basic|developing|strong|leading",
            "rationale": "string",
            "evidence_quotes": ["short direct snippets"],
        },
        "strategy_resilience": {
            "score_0_to_4": "number",
            "rating": "insufficient|basic|developing|strong|leading",
            "rationale": "string",
            "evidence_quotes": ["short direct snippets"],
        },
        "risk_management_integration": {
            "score_0_to_4": "number",
            "rating": "insufficient|basic|developing|strong|leading",
            "rationale": "string",
            "evidence_quotes": ["short direct snippets"],
        },
        "metrics_targets_quality": {
            "score_0_to_4": "number",
            "rating": "insufficient|basic|developing|strong|leading",
            "rationale": "string",
            "evidence_quotes": ["short direct snippets"],
        },
        "transition_plan_credibility": {
            "score_0_to_4": "number",
            "rating": "insufficient|basic|developing|strong|leading",
            "rationale": "string",
            "evidence_quotes": ["short direct snippets"],
        },
    },
    "base_weighted_score_100": "number",
    "overall_score_100": "number",
    "overall_grade": "AAA|AA|A|BBB|BB|B|CCC",
    "quality_adjustments": {
        "score_adjustment_points": "number",
        "evidence_coverage_ratio": "number",
        "adjustment_reasons": ["string"],
    },
    "key_strengths": ["string"],
    "key_gaps": ["string"],
    "semantic_analysis": {
        "commitment_tone": "aspirational|balanced|execution-oriented",
        "specificity_level": "low|medium|high",
        "delivery_signal": "low|medium|high",
        "greenwashing_risk": "low|medium|high",
        "narrative_consistency": "low|medium|high",
        "distinctive_language": ["short phrase"],
    },
    "confidence": "low|medium|high",
    "notes": "string",
}

def _indicator_keys() -> List[str]:
    return list(TCFD_COMMITMENT_RUBRIC.keys())


def _grade_from_score(score_100: float) -> str:
    if score_100 >= 90:
        return "AAA"
    if score_100 >= 80:
        return "AA"
    if score_100 >= 70:
        return "A"
    if score_100 >= 60:
        return "BBB"
    if score_100 >= 50:
        return "BB"
    if score_100 >= 40:
        return "B"
    return "CCC"


def _safe_float(x: Any) -> float | None:
    try:
        v = float(x)
    except Exception:
        return None
    if v < 0:
        v = 0.0
    if v > 4:
        v = 4.0
    return v


def _build_rubric_block() -> str:
    principle_block = "\n".join(
        [f"- P{i+1}: {p}" for i, p in enumerate(TCFD_DISCLOSURE_QUALITY_PRINCIPLES)]
    )

    blocks = []
    for k, spec in TCFD_COMMITMENT_RUBRIC.items():
        levels = spec.get("score_levels", {})
        required_items = spec.get("required_disclosures", [])
        red_flags = spec.get("over_scoring_red_flags", [])

        required_text = "\n".join([f"  - {item}" for item in required_items])
        red_flag_text = "\n".join([f"  - {item}" for item in red_flags])

        blocks.append(
            f"""
Indicator: {k}
TCFD reference: {spec['tcfd_ref']}
IFRS S2 reference: {spec.get('ifrs_s2_ref', 'n/a')}
Weight: {spec['weight']}
What to check: {spec['what_to_check']}
Must-have evidence checklist:
{required_text}
Over-scoring red flags:
{red_flag_text}
Scoring guide:
0 = {levels.get(0,'')}
1 = {levels.get(1,'')}
2 = {levels.get(2,'')}
3 = {levels.get(3,'')}
4 = {levels.get(4,'')}
""".strip()
        )

    return (
        "TCFD disclosure quality principles (cross-cutting):\n"
        + principle_block
        + "\n\n"
        + "\n\n".join(blocks)
    )

def _apply_quality_adjustments(data: Dict[str, Any]) -> Dict[str, Any]:
    reasons: List[str] = []
    score_adjustment = 0.0

    indicator_scores = data.get("indicator_scores", {}) or {}
    total_indicators = max(1, len(TCFD_COMMITMENT_RUBRIC))
    indicators_with_any_quote = 0

    for k in _indicator_keys():
        node = indicator_scores.get(k, {}) or {}
        s = node.get("score_0_to_4")
        quotes = [q for q in (node.get("evidence_quotes") or []) if isinstance(q, str) and q.strip()]

        if quotes:
            indicators_with_any_quote += 1

        if isinstance(s, (int, float)):
            if s >= 3.0 and len(quotes) == 0:
                score_adjustment -= 8.0
                reasons.append(f"{k}: high score without evidence quote support")
            elif s >= 3.5 and len(quotes) < 2:
                score_adjustment -= 3.0
                reasons.append(f"{k}: very high score with limited evidence quotes")

            rationale = str(node.get("rationale", "")).lower()
            has_aspirational = bool(re.search(r"aim|aspire|intend|plan to|seek to|strive|ambition", rationale))
            has_delivery = bool(re.search(r"implemented|launched|achieved|reduced|invested|completed|milestone|progress", rationale))
            if s >= 3.0 and has_aspirational and not has_delivery:
                score_adjustment -= 2.0
                reasons.append(f"{k}: rationale appears aspirational without delivery wording")

    evidence_coverage_ratio = round(indicators_with_any_quote / total_indicators, 3)
    if evidence_coverage_ratio < 0.6:
        score_adjustment -= 4.0
        reasons.append("cross-indicator evidence coverage is below 60%")

    sem = data.get("semantic_analysis", {}) or {}
    if sem.get("specificity_level") == "low":
        score_adjustment -= 4.0
        reasons.append("semantic signal: low specificity")
    if sem.get("narrative_consistency") == "low":
        score_adjustment -= 4.0
        reasons.append("semantic signal: low narrative consistency")
    if sem.get("greenwashing_risk") == "high":
        score_adjustment -= 6.0
        reasons.append("semantic signal: high greenwashing risk")

    if (
        sem.get("delivery_signal") == "high"
        and sem.get("specificity_level") == "high"
        and sem.get("greenwashing_risk") in {"low", "medium"}
    ):
        score_adjustment += 2.0
        reasons.append("semantic signal: strong delivery and specificity")

    return {
        "score_adjustment_points": round(score_adjustment, 1),
        "evidence_coverage_ratio": evidence_coverage_ratio,
        "adjustment_reasons": reasons,
    }


def _commitment_signal_score(indicator_key: str, text: str) -> float:
    spec = TCFD_COMMITMENT_RUBRIC[indicator_key]
    lower = text.lower()
    score = 0.0
    # indicator keywords
    for term in spec.get("retrieval_terms", []):
        if term.lower() in lower:
            score += 0.2
    # execution language
    if re.search(r"implemented|launched|reduced|invested|achieved|completed|progress", lower):
        score += 0.6
    # plan / target language
    if re.search(r"target|goal|net zero|by 2030|by 2040|by 2050|baseline", lower):
        score += 0.5
    # governance/risk language
    if re.search(r"board|committee|oversight|risk management|scenario", lower):
        score += 0.4
    return score


def _semantic_feature_snapshot(chunks: List[Dict[str, Any]]) -> Dict[str, Any]:
    text = " ".join([c.get("text", "") for c in chunks]).lower()
    aspirational = len(re.findall(r"aim|aspire|intend|plan to|seek to|strive", text))
    execution = len(re.findall(r"implemented|deployed|reduced|achieved|invested|completed", text))
    methodology = len(re.findall(r"ghg protocol|market-based|location-based|boundary|methodology", text))
    return {
        "aspirational_term_count": aspirational,
        "execution_term_count": execution,
        "methodology_term_count": methodology,
    }


def _format_commitment_chunks(chunks: List[Dict[str, Any]], max_chars: int = 680) -> str:
    if not chunks:
        return "<no chunks>"
    blocks = []
    for i, c in enumerate(chunks, 1):
        rr = c.get("rerank_score")
        rr_txt = f"{rr:.4f}" if isinstance(rr, (int, float)) else "n/a"
        blocks.append(
            f"[Chunk {i}] file={c['file_name']} chunk={c['chunk_id']} sim={c['similarity']:.4f} rerank={rr_txt}\\n"
            f"{c['text'][:max_chars]}"
        )
    return "\\n\\n".join(blocks)


def _build_commitment_hyde_prompt(company_name: str, indicator_key: str) -> str:
    spec = TCFD_COMMITMENT_RUBRIC[indicator_key]
    return f"""
Company: {company_name}
Indicator: {indicator_key}
TCFD reference: {spec['tcfd_ref']}
What to check: {spec['what_to_check']}
Likely terms: {', '.join(spec.get('retrieval_terms', []))}

Write 3 short hypothetical ESG-report style lines that would likely contain evidence for this indicator.
Use policy + implementation language.
Do not output JSON.
""".strip()


def _build_commitment_query(company_name: str, indicator_key: str, hyde_text: str = "") -> str:
    spec = TCFD_COMMITMENT_RUBRIC[indicator_key]
    base = f"""
Retrieve evidence for {company_name} on indicator {indicator_key}.
TCFD reference: {spec['tcfd_ref']}
Focus: {spec['what_to_check']}
Keywords: {' '.join(spec.get('retrieval_terms', []))}
""".strip()
    if hyde_text:
        base += "\n\nHypothetical evidence phrasing:\n" + hyde_text
    return base


def _build_commitment_table_query(company_name: str, indicator_key: str) -> str:
    spec = TCFD_COMMITMENT_RUBRIC[indicator_key]
    return f"""
Company {company_name} report sections for {indicator_key}.
Find sections and tables likely titled: governance, risk management, targets, progress, transition, metrics.
Look for year references, milestone language, baseline year, and implementation progress.
Keywords: {' '.join(spec.get('retrieval_terms', []))}
""".strip()


def _normalize_commitment_output(data: Dict[str, Any], company_name: str) -> Dict[str, Any]:
    if not isinstance(data, dict):
        data = {}
    data.setdefault("company", company_name)

    indicator_scores = data.get("indicator_scores")
    if not isinstance(indicator_scores, dict):
        indicator_scores = {}

    normalized_scores: Dict[str, Dict[str, Any]] = {}
    weighted = 0.0

    for k, spec in TCFD_COMMITMENT_RUBRIC.items():
        node = indicator_scores.get(k, {})
        if not isinstance(node, dict):
            node = {}

        s = _safe_float(node.get("score_0_to_4"))
        node["score_0_to_4"] = s
        if not isinstance(node.get("evidence_quotes"), list):
            node["evidence_quotes"] = []

        normalized_scores[k] = node

        if s is not None:
            weighted += (s / 4.0) * 100.0 * float(spec.get("weight", 0.0))

    data["indicator_scores"] = normalized_scores

    base_score = round(weighted, 1)
    qa = _apply_quality_adjustments(data)
    adjusted_score = max(0.0, min(100.0, base_score + float(qa.get("score_adjustment_points", 0.0))))

    data["base_weighted_score_100"] = base_score
    data["overall_score_100"] = round(adjusted_score, 1)
    data["overall_grade"] = _grade_from_score(float(data["overall_score_100"]))
    data["quality_adjustments"] = qa

    if not isinstance(data.get("key_strengths"), list):
        data["key_strengths"] = []
    if not isinstance(data.get("key_gaps"), list):
        data["key_gaps"] = []
    if not isinstance(data.get("semantic_analysis"), dict):
        data["semantic_analysis"] = {}

    data.setdefault("confidence", "medium")
    data.setdefault("notes", "")
    return data


def _build_commitment_prompt(company_name: str, retrieval_package: Dict[str, Any]) -> str:
    indicator_blocks = []
    for k, spec in TCFD_COMMITMENT_RUBRIC.items():
        section = retrieval_package["by_indicator"].get(k, {})
        required_items = spec.get("required_disclosures", [])
        red_flags = spec.get("over_scoring_red_flags", [])

        indicator_blocks.append(
            f"""
Indicator: {k}
TCFD reference: {spec['tcfd_ref']}
IFRS S2 reference: {spec.get('ifrs_s2_ref', 'n/a')}
What to check: {spec['what_to_check']}
Required checklist:
{chr(10).join(['- ' + x for x in required_items])}
Over-scoring red flags:
{chr(10).join(['- ' + x for x in red_flags])}
Retrieved evidence:
{_format_commitment_chunks(section.get('hits', []))}
""".strip()
        )

    scoring_rules = """
Scoring rules (strict):
1. Use only provided excerpts.
2. Score each indicator from 0 to 4 using the rubric anchors.
3. For score >= 3, include at least two concrete evidence quotes if possible.
4. If disclosures are aspirational without implementation evidence, cap score at 2.
5. If required checklist elements are mostly missing, score conservatively.
6. Keep rationales evidence-based and concise.
7. Provide semantic analysis fields exactly as requested.
8. Return JSON only.
""".strip()

    return f"""
Task: Evaluate climate-commitment disclosure quality for {company_name}.
Framework alignment: TCFD structure + IFRS S2-oriented climate disclosure expectations.

Output JSON schema:
{json.dumps(COMMITMENT_OUTPUT_SCHEMA, indent=2)}

Rubric:
{_build_rubric_block()}

Evidence by indicator:
{chr(10).join([''] + [b + chr(10) for b in indicator_blocks])}

{scoring_rules}
""".strip()


In [ ]:
def extract_first_json_object(text: str) -> str:
    text = text.strip()
    if text.startswith('{') and text.endswith('}'):
        return text

    start = text.find('{')
    if start == -1:
        raise ValueError('No JSON object start found')

    depth = 0
    for i in range(start, len(text)):
        ch = text[i]
        if ch == '{':
            depth += 1
        elif ch == '}':
            depth -= 1
            if depth == 0:
                return text[start:i+1]
    raise ValueError('No balanced JSON object found')


def parse_model_json(raw_text: str) -> Dict[str, Any]:
    try:
        return json.loads(raw_text)
    except Exception:
        candidate = extract_first_json_object(raw_text)
        return json.loads(candidate)


HYDE_SYSTEM_PROMPT = """
You write short hypothetical ESG report excerpts to improve retrieval.
Write in report language (tables, footnotes, methodology text).
Do not mention uncertainty, do not add commentary, and do not output JSON.
""".strip()


In [ ]:
COMMITMENT_HYDE_CACHE: Dict[str, str] = {}


def generate_commitment_hyde(company_name: str, indicator_key: str, model_name: str | None = None) -> str:
    model_name = model_name or os.getenv("COLAB_AI_MODEL_NAME", "google/gemini-2.5-pro")
    cache_key = f"{company_name}::{indicator_key}::{model_name}"
    if cache_key in COMMITMENT_HYDE_CACHE:
        return COMMITMENT_HYDE_CACHE[cache_key]

    prompt = _build_commitment_hyde_prompt(company_name, indicator_key)
    txt = call_llm(system_prompt=HYDE_SYSTEM_PROMPT, user_prompt=prompt, model_name=model_name)
    txt = re.sub(r"\s+", " ", txt).strip()
    COMMITMENT_HYDE_CACHE[cache_key] = txt
    return txt


def retrieve_commitment_context(
    doc: Dict[str, Any],
    top_k_per_indicator: int = 4,
    retrieval_pool_size_per_indicator: int = 12,
    use_hyde_retrieval: bool = True,
    hyde_model_name: str | None = None,
    use_table_query_retrieval: bool = True,
    commitment_signal_weight: float = 0.08,
    table_signal_weight: float = 0.04,
) -> Dict[str, Any]:
    by_indicator: Dict[str, Dict[str, Any]] = {}
    merged_map: Dict[str, Dict[str, Any]] = {}
    hyde_errors: Dict[str, str] = {}

    for indicator_key in _indicator_keys():
        hyde_text = ""
        if use_hyde_retrieval:
            try:
                hyde_text = generate_commitment_hyde(doc["company"], indicator_key, model_name=hyde_model_name)
            except Exception as e:
                hyde_errors[indicator_key] = str(e)

        semantic_query = _build_commitment_query(doc["company"], indicator_key, hyde_text=hyde_text)
        table_query = _build_commitment_table_query(doc["company"], indicator_key)
        query_texts = [semantic_query]
        if use_table_query_retrieval:
            query_texts.append(table_query)

        pooled_hits = retrieve_chunks_multiquery(
            query_texts=query_texts,
            top_k_per_query=retrieval_pool_size_per_indicator,
            filter_doc_id=doc["doc_id"],
        )

        ranked_hits: List[Dict[str, Any]] = []
        for h in pooled_hits:
            item = dict(h)
            text = item.get("text", "")
            c_signal = _commitment_signal_score(indicator_key, text)
            t_signal = compute_table_signal_score(text)
            rr = float(item.get("similarity", 0.0))
            rr += commitment_signal_weight * c_signal
            rr += table_signal_weight * t_signal
            item["commitment_signal_score"] = round(c_signal, 3)
            item["table_signal_score"] = round(t_signal, 3)
            item["rerank_score"] = round(rr, 6)
            ranked_hits.append(item)

        ranked_hits.sort(key=lambda x: x["rerank_score"], reverse=True)
        final_hits = ranked_hits[:top_k_per_indicator]

        by_indicator[indicator_key] = {
            "query_texts": query_texts,
            "hyde_text": hyde_text,
            "hits": final_hits,
        }

        for h in final_hits:
            key = f"{h['file_name']}::{h['chunk_id']}"
            prev = merged_map.get(key)
            if prev is None or h["rerank_score"] > prev.get("rerank_score", 0.0):
                merged_map[key] = h

    merged_hits = sorted(merged_map.values(), key=lambda x: x.get("rerank_score", x.get("similarity", 0.0)), reverse=True)
    return {
        "by_indicator": by_indicator,
        "merged_hits": merged_hits,
        "hyde_errors": hyde_errors,
        "semantic_feature_snapshot": _semantic_feature_snapshot(merged_hits[:16]),
    }




def build_retrieval_debug(
    by_indicator: Dict[str, Dict[str, Any]],
    max_hits_per_indicator: int = 3,
    snippet_chars: int = 240,
) -> Dict[str, Any]:
    out: Dict[str, Any] = {}
    for indicator_key, payload in (by_indicator or {}).items():
        q_texts = payload.get("query_texts", [])
        compact_hits = []
        for h in (payload.get("hits", []) or [])[:max_hits_per_indicator]:
            compact_hits.append(
                {
                    "chunk_id": h.get("chunk_id"),
                    "similarity": round(float(h.get("similarity", 0.0)), 4),
                    "rerank_score": round(float(h.get("rerank_score", h.get("similarity", 0.0))), 4),
                    "commitment_signal_score": h.get("commitment_signal_score"),
                    "table_signal_score": h.get("table_signal_score"),
                    "snippet": re.sub(r"\s+", " ", (h.get("text") or "")).strip()[:snippet_chars],
                }
            )

        out[indicator_key] = {
            "query_texts": q_texts,
            "hits": compact_hits,
        }
    return out

def commitment_score_for_doc(
    doc: Dict[str, Any],
    model_name: str | None = None,
    top_k_per_indicator: int = 4,
    retrieval_pool_size_per_indicator: int = 12,
    use_hyde_retrieval: bool = True,
    hyde_model_name: str | None = None,
    use_table_query_retrieval: bool = True,
    commitment_signal_weight: float = 0.08,
    table_signal_weight: float = 0.04,
) -> Dict[str, Any]:
    retrieval_package = retrieve_commitment_context(
        doc,
        top_k_per_indicator=top_k_per_indicator,
        retrieval_pool_size_per_indicator=retrieval_pool_size_per_indicator,
        use_hyde_retrieval=use_hyde_retrieval,
        hyde_model_name=hyde_model_name,
        use_table_query_retrieval=use_table_query_retrieval,
        commitment_signal_weight=commitment_signal_weight,
        table_signal_weight=table_signal_weight,
    )

    if not retrieval_package["merged_hits"]:
        return {
            "company": doc["company"],
            "error": "No retrieved commitment evidence",
            "_rag_meta": {
                "provider": "colab_ai",
                "doc_id": doc["doc_id"],
                "file_name": doc["file_name"],
                "source_type": doc["source_type"],
            },
        }

    user_prompt = _build_commitment_prompt(doc["company"], retrieval_package)
    t0 = time.time()
    raw = call_llm(system_prompt=COMMITMENT_SYSTEM_PROMPT, user_prompt=user_prompt, model_name=model_name)
    elapsed = time.time() - t0

    try:
        parsed = parse_model_json(raw)
    except Exception as e:
        parsed = {
            "company": doc["company"],
            "parse_error": str(e),
            "raw_model_output": raw[:8000],
        }

    data = _normalize_commitment_output(parsed, company_name=doc["company"])

    requested_model = model_name or os.getenv("COLAB_AI_MODEL_NAME", "google/gemini-2.5-pro")
    actual_model = _COLAB_AI_CACHE.get("last_model_used") or requested_model

    indicator_hit_counts = {
        k: len(v.get("hits", [])) for k, v in retrieval_package["by_indicator"].items()
    }

    data["_rag_meta"] = {
        "provider": "colab_ai",
        "model": actual_model,
        "requested_model": requested_model,
        "model_attempts": _COLAB_AI_CACHE.get("last_model_attempts", []),
        "doc_id": doc["doc_id"],
        "file_name": doc["file_name"],
        "source_type": doc["source_type"],
        "top_k_per_indicator": top_k_per_indicator,
        "retrieval_pool_size_per_indicator": retrieval_pool_size_per_indicator,
        "use_hyde_retrieval": use_hyde_retrieval,
        "hyde_model_name": hyde_model_name or requested_model,
        "use_table_query_retrieval": use_table_query_retrieval,
        "commitment_signal_weight": commitment_signal_weight,
        "table_signal_weight": table_signal_weight,
        "indicator_hit_counts": indicator_hit_counts,
        "semantic_feature_snapshot": retrieval_package.get("semantic_feature_snapshot", {}),
        "latency_sec": round(elapsed, 2),
    }
    data["_retrieval_debug"] = build_retrieval_debug(retrieval_package.get("by_indicator", {}))
    return data


## Step 7. Run Commitment Scoring (One Company + Batch)

We first run one company, then score the four companies with the same rubric.


In [ ]:
COMMITMENT_TOP_K_PER_INDICATOR = 4
COMMITMENT_RETRIEVAL_POOL_SIZE_PER_INDICATOR = 12
COMMITMENT_USE_HYDE_RETRIEVAL = True
COMMITMENT_HYDE_MODEL_NAME = "google/gemini-2.5-flash-lite"
COMMITMENT_USE_TABLE_QUERY_RETRIEVAL = True
COMMITMENT_SIGNAL_WEIGHT = 0.08
COMMITMENT_TABLE_SIGNAL_WEIGHT = 0.04

RUN_COMMITMENT_ONE = bool(IN_COLAB)
RUN_COMMITMENT_BATCH = bool(IN_COLAB)

commitment_single_result = None
if RUN_COMMITMENT_ONE and documents:
    preferred_order = ["Amazon", "Google", "Apple", "Meta"]
    selected = None
    for name in preferred_order:
        selected = next((d for d in documents if d["company"] == name), None)
        if selected:
            break

    print("Running commitment scoring for:", selected["company"], "|", selected["file_name"])
    commitment_single_result = commitment_score_for_doc(
        selected,
        model_name=COLAB_AI_MODEL_NAME,
        top_k_per_indicator=COMMITMENT_TOP_K_PER_INDICATOR,
        retrieval_pool_size_per_indicator=COMMITMENT_RETRIEVAL_POOL_SIZE_PER_INDICATOR,
        use_hyde_retrieval=COMMITMENT_USE_HYDE_RETRIEVAL,
        hyde_model_name=COMMITMENT_HYDE_MODEL_NAME,
        use_table_query_retrieval=COMMITMENT_USE_TABLE_QUERY_RETRIEVAL,
        commitment_signal_weight=COMMITMENT_SIGNAL_WEIGHT,
        table_signal_weight=COMMITMENT_TABLE_SIGNAL_WEIGHT,
    )
    print(json.dumps(commitment_single_result, indent=2, ensure_ascii=False)[:5000])
else:
    print("Skipping commitment one-company run (not in Colab or no documents).")


commitment_results: List[Dict[str, Any]] = []
if RUN_COMMITMENT_BATCH:
    for doc in documents:
        if doc["company"] not in {"Amazon", "Google", "Apple", "Meta"}:
            continue
        print(f"\n=== Commitment scoring: {doc['company']} | {doc['file_name']} ===")
        try:
            r = commitment_score_for_doc(
                doc,
                model_name=COLAB_AI_MODEL_NAME,
                top_k_per_indicator=COMMITMENT_TOP_K_PER_INDICATOR,
                retrieval_pool_size_per_indicator=COMMITMENT_RETRIEVAL_POOL_SIZE_PER_INDICATOR,
                use_hyde_retrieval=COMMITMENT_USE_HYDE_RETRIEVAL,
                hyde_model_name=COMMITMENT_HYDE_MODEL_NAME,
                use_table_query_retrieval=COMMITMENT_USE_TABLE_QUERY_RETRIEVAL,
                commitment_signal_weight=COMMITMENT_SIGNAL_WEIGHT,
                table_signal_weight=COMMITMENT_TABLE_SIGNAL_WEIGHT,
            )
        except Exception as e:
            r = {
                "company": doc["company"],
                "error": str(e),
                "_rag_meta": {
                    "provider": "colab_ai",
                    "doc_id": doc["doc_id"],
                    "file_name": doc["file_name"],
                    "source_type": doc["source_type"],
                },
            }
        commitment_results.append(r)

    print("\nCommitment batch complete:", len(commitment_results), "result(s)")
else:
    print("RUN_COMMITMENT_BATCH = False (set Colab runtime to run this stage).")


## Step 8. Build Commitment Score Table


In [ ]:
def _get_indicator_score(row: Dict[str, Any], key: str) -> float | None:
    node = (row.get("indicator_scores") or {}).get(key, {})
    try:
        if node.get("score_0_to_4") is None:
            return None
        return float(node.get("score_0_to_4"))
    except Exception:
        return None


commitment_summary_df = pd.DataFrame(
    [
        {
            "company": r.get("company"),
            "base_weighted_score_100": r.get("base_weighted_score_100"),
            "quality_adjustment_points": ((r.get("quality_adjustments") or {}).get("score_adjustment_points")),
            "evidence_coverage_ratio": ((r.get("quality_adjustments") or {}).get("evidence_coverage_ratio")),
            "overall_score_100": r.get("overall_score_100"),
            "overall_grade": r.get("overall_grade"),
            "governance_oversight_score_0_4": _get_indicator_score(r, "governance_oversight"),
            "strategy_resilience_score_0_4": _get_indicator_score(r, "strategy_resilience"),
            "risk_management_integration_score_0_4": _get_indicator_score(r, "risk_management_integration"),
            "metrics_targets_quality_score_0_4": _get_indicator_score(r, "metrics_targets_quality"),
            "transition_plan_credibility_score_0_4": _get_indicator_score(r, "transition_plan_credibility"),
            "tone": ((r.get("semantic_analysis") or {}).get("commitment_tone")),
            "specificity_level": ((r.get("semantic_analysis") or {}).get("specificity_level")),
            "delivery_signal": ((r.get("semantic_analysis") or {}).get("delivery_signal")),
            "greenwashing_risk": ((r.get("semantic_analysis") or {}).get("greenwashing_risk")),
            "confidence": r.get("confidence"),
            "error": r.get("error"),
            "parse_error": r.get("parse_error"),
        }
        for r in commitment_results
    ]
)

commitment_summary_df


## Step 9. Save Commitment Scoring Outputs


In [ ]:
commitment_timestamp = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")

commitment_json_path = OUTPUT_DIR / "climate_commitment_scores.json"
commitment_csv_path = OUTPUT_DIR / "climate_commitment_scores.csv"
commitment_run_record_path = OUTPUT_DIR / f"climate_commitment_run_record_{commitment_timestamp}.json"

with open(commitment_json_path, "w", encoding="utf-8") as f:
    json.dump(commitment_results, f, indent=2, ensure_ascii=False)

commitment_summary_df.to_csv(commitment_csv_path, index=False)

commitment_run_record = {
    "run_timestamp_utc": commitment_timestamp,
    "provider": "colab_ai",
    "model": COLAB_AI_MODEL_NAME,
    "model_fallbacks": COLAB_AI_FALLBACK_MODELS,
    "rubric_indicators": _indicator_keys(),
    "tcfd_alignment": {k: v["tcfd_ref"] for k, v in TCFD_COMMITMENT_RUBRIC.items()},
    "top_k_per_indicator": COMMITMENT_TOP_K_PER_INDICATOR,
    "retrieval_pool_size_per_indicator": COMMITMENT_RETRIEVAL_POOL_SIZE_PER_INDICATOR,
    "use_hyde_retrieval": COMMITMENT_USE_HYDE_RETRIEVAL,
    "hyde_model_name": COMMITMENT_HYDE_MODEL_NAME,
    "use_table_query_retrieval": COMMITMENT_USE_TABLE_QUERY_RETRIEVAL,
    "commitment_signal_weight": COMMITMENT_SIGNAL_WEIGHT,
    "table_signal_weight": COMMITMENT_TABLE_SIGNAL_WEIGHT,
    "target_companies": ["Amazon", "Google", "Apple", "Meta"],
    "doc_count_loaded": len(documents),
    "doc_files": [d["file_name"] for d in documents],
    "vector_collection": COLLECTION_NAME,
    "embedding_model": EMBED_MODEL_NAME,
    "commitment_json_path": str(commitment_json_path),
    "commitment_csv_path": str(commitment_csv_path),
}

with open(commitment_run_record_path, "w", encoding="utf-8") as f:
    json.dump(commitment_run_record, f, indent=2, ensure_ascii=False)

print("Saved:", commitment_json_path)
print("Saved:", commitment_csv_path)
print("Saved:", commitment_run_record_path)


## Step 10. Visualize Context Understanding

We can evaluate context understanding from three angles:
- retrieval coverage: how many relevant chunks were found per indicator
- evidence usage: how many evidence quotes the model used per indicator
- indicator quality score: final rubric score per indicator

Interpretation guide:
- high score + high evidence usage usually means strong grounded understanding
- high score + low evidence usage means We should inspect grounding quality
- low score + high retrieval coverage means retrieval is broad, but evidence quality may be weak


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

INDICATOR_ORDER = list(TCFD_COMMITMENT_RUBRIC.keys())
INDICATOR_LABELS = {k: k.replace("_", " ").title() for k in INDICATOR_ORDER}


def build_context_viz_df(results: List[Dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for r in results:
        company = r.get("company")
        if not company:
            continue

        scores = r.get("indicator_scores") or {}
        hit_counts = ((r.get("_rag_meta") or {}).get("indicator_hit_counts") or {})

        for k in INDICATOR_ORDER:
            node = scores.get(k) or {}
            raw_score = node.get("score_0_to_4")
            try:
                score = float(raw_score) if raw_score is not None else np.nan
            except Exception:
                score = np.nan

            quotes = node.get("evidence_quotes") or []
            quote_count = sum(1 for q in quotes if isinstance(q, str) and q.strip())
            hits = int(hit_counts.get(k, 0) or 0)
            evidence_per_hit = quote_count / max(1, hits)

            rows.append(
                {
                    "company": company,
                    "indicator_key": k,
                    "indicator_label": INDICATOR_LABELS[k],
                    "score_0_4": score,
                    "retrieved_hits": hits,
                    "evidence_quotes_used": quote_count,
                    "evidence_per_hit": evidence_per_hit,
                }
            )

    return pd.DataFrame(rows)


def plot_heatmap(mat: pd.DataFrame, title: str, cmap: str, vmin: float | None, vmax: float | None, fmt: str):
    if mat.empty:
        print(f"Skip {title}: empty matrix")
        return

    values = mat.to_numpy(dtype=float)
    fig_w = max(8, 1.6 * mat.shape[1] + 2)
    fig_h = max(4, 0.9 * mat.shape[0] + 1.5)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    im = ax.imshow(values, cmap=cmap, aspect="auto", vmin=vmin, vmax=vmax)

    ax.set_xticks(range(mat.shape[1]))
    ax.set_yticks(range(mat.shape[0]))
    ax.set_xticklabels(mat.columns, rotation=35, ha="right")
    ax.set_yticklabels(mat.index)
    ax.set_title(title)

    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            val = values[i, j]
            txt = "-" if np.isnan(val) else fmt.format(val)
            ax.text(j, i, txt, ha="center", va="center", fontsize=9, color="black")

    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    plt.tight_layout()
    plt.show()


context_viz_df = build_context_viz_df(commitment_results)
if context_viz_df.empty:
    print("No commitment results found. Run Step 7 first.")
else:
    company_order = []
    if "company" in commitment_summary_df.columns:
        for c in commitment_summary_df["company"].tolist():
            if isinstance(c, str) and c not in company_order:
                company_order.append(c)
    if not company_order:
        company_order = sorted(context_viz_df["company"].dropna().unique().tolist())

    indicator_order = [INDICATOR_LABELS[k] for k in INDICATOR_ORDER]

    score_mat = (
        context_viz_df.pivot(index="company", columns="indicator_label", values="score_0_4")
        .reindex(index=company_order, columns=indicator_order)
    )
    hit_mat = (
        context_viz_df.pivot(index="company", columns="indicator_label", values="retrieved_hits")
        .reindex(index=company_order, columns=indicator_order)
    )
    eph_mat = (
        context_viz_df.pivot(index="company", columns="indicator_label", values="evidence_per_hit")
        .reindex(index=company_order, columns=indicator_order)
    )

    plot_heatmap(score_mat, "LLM Indicator Score (0-4)", cmap="YlGn", vmin=0, vmax=4, fmt="{:.1f}")
    plot_heatmap(hit_mat, "Retrieved Chunks per Indicator", cmap="Blues", vmin=0, vmax=max(1.0, np.nanmax(hit_mat.to_numpy(dtype=float))), fmt="{:.0f}")
    plot_heatmap(eph_mat, "Evidence Quotes per Retrieved Chunk", cmap="Oranges", vmin=0, vmax=max(1.0, np.nanmax(eph_mat.to_numpy(dtype=float))), fmt="{:.2f}")

    company_scatter = (
        context_viz_df.groupby("company", as_index=False)
        .agg(
            avg_indicator_score=("score_0_4", "mean"),
            avg_retrieved_hits=("retrieved_hits", "mean"),
            avg_evidence_per_hit=("evidence_per_hit", "mean"),
        )
        .sort_values("avg_indicator_score", ascending=False)
    )

    fig, ax = plt.subplots(figsize=(8, 5))
    sc = ax.scatter(
        company_scatter["avg_retrieved_hits"],
        company_scatter["avg_indicator_score"],
        c=company_scatter["avg_evidence_per_hit"],
        cmap="viridis",
        s=140,
        edgecolor="black",
        linewidth=0.7,
    )

    for _, row in company_scatter.iterrows():
        ax.text(row["avg_retrieved_hits"] + 0.03, row["avg_indicator_score"] + 0.02, row["company"], fontsize=9)

    ax.set_xlabel("Average Retrieved Chunks per Indicator")
    ax.set_ylabel("Average Indicator Score (0-4)")
    ax.set_title("Company-Level Context Understanding Map")
    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label("Average Evidence Quotes per Retrieved Chunk")
    plt.tight_layout()
    plt.show()

    context_viz_df


## Step 11. Inspect Retrieval Evidence for One Indicator

We can inspect one company and one indicator directly.
This helps us verify whether the retrieved context actually supports the final scoring rationale.


In [ ]:
def build_retrieval_preview_df(results: List[Dict[str, Any]], company_name: str, indicator_key: str) -> tuple[pd.DataFrame, List[str]]:
    target = next((r for r in results if r.get("company") == company_name), None)
    if not target:
        return pd.DataFrame(), []

    debug = target.get("_retrieval_debug") or {}
    payload = debug.get(indicator_key) or {}
    hits = payload.get("hits") or []

    rows = []
    for h in hits:
        rows.append(
            {
                "chunk_id": h.get("chunk_id"),
                "similarity": h.get("similarity"),
                "rerank_score": h.get("rerank_score"),
                "commitment_signal_score": h.get("commitment_signal_score"),
                "table_signal_score": h.get("table_signal_score"),
                "snippet": h.get("snippet"),
            }
        )

    return pd.DataFrame(rows), payload.get("query_texts") or []


if not commitment_results:
    print("No commitment results found. Run Step 7 first.")
else:
    default_company = commitment_results[0].get("company", "Amazon")
    selected_company = default_company
    selected_indicator = "metrics_targets_quality"

    preview_df, query_texts = build_retrieval_preview_df(commitment_results, selected_company, selected_indicator)

    print("Selected company:", selected_company)
    print("Selected indicator:", selected_indicator)
    print("Query text count:", len(query_texts))
    for i, q in enumerate(query_texts, 1):
        print(f"- Query {i}: {q[:220]}")

    if preview_df.empty:
        print("No retrieval preview found. If Step 7 was executed before this update, rerun Step 7 once.")
    else:
        preview_df
